In [1]:
import pandas as pd
import numpy as np
import os

# Load the cleaned data
df = pd.read_csv('/Users/unnatiagarwal/Documents/Semester_4/Applied_Project/Untitled/Cleaning_data_cleaned.csv')

# Create output directory for categorized CSVs
output_dir = 'categorized_diabetes'
os.makedirs(output_dir, exist_ok=True)

# Define categorization functions
def is_absolute_diagnosed_diabetes(row):
    """DIQ010 = 1(Yes) & LBXGH ≥6.5"""
    return (row['DIQ010'] == 1) & (row['LBXGH'] >= 6.5)

def is_absolute_prediabetes(row):
    """DIQ160 = 1(Yes) & LBXGH = 5.7-6.49"""
    return (row['DIQ160'] == 1) & (row['LBXGH'] >= 5.7) & (row['LBXGH'] < 6.5)

def is_absolute_no_diabetes(row):
    """DIQ010=2(No) & LBXGH>5.7 OR DIQ160=2(No) & LBXGH >5.7"""
    condition1 = (row['DIQ010'] == 2) & (row['LBXGH'] > 5.7)
    condition2 = (row['DIQ160'] == 2) & (row['LBXGH'] > 5.7)
    return condition1 | condition2

def is_wrongly_diagnosed(row):
    """Multiple conditions for wrongly diagnosed"""
    conditions = []

    # DIQ010=1(YES) & LBXGH<6.5
    conditions.append((row['DIQ010'] == 1) & (row['LBXGH'] < 6.5))

    # DIQ010=2(No) & LBXGH>6.5
    conditions.append((row['DIQ010'] == 2) & (row['LBXGH'] > 6.5))

    # DIQ010=3(Borderline) & (LBXGH<5.7 or >6.49)
    conditions.append((row['DIQ010'] == 3) & ((row['LBXGH'] < 5.7) | (row['LBXGH'] > 6.49)))

    # DIQ160=1(YES) & (LBXGH<5.7 or >6.49)
    conditions.append((row['DIQ160'] == 1) & ((row['LBXGH'] < 5.7) | (row['LBXGH'] > 6.49)))

    # DIQ160=2(No) & LBXGH>5.7 & <6.5
    conditions.append((row['DIQ160'] == 2) & (row['LBXGH'] > 5.7) & (row['LBXGH'] < 6.5))

    return any(conditions)

# Apply categorizations
df['absolute_diagnosed_diabetes'] = df.apply(is_absolute_diagnosed_diabetes, axis=1)
df['absolute_prediabetes'] = df.apply(is_absolute_prediabetes, axis=1)
df['absolute_no_diabetes'] = df.apply(is_absolute_no_diabetes, axis=1)
df['wrongly_diagnosed'] = df.apply(is_wrongly_diagnosed, axis=1)

# Create separate DataFrames for each category
absolute_diagnosed_diabetes_df = df[df['absolute_diagnosed_diabetes']].drop(['absolute_diagnosed_diabetes', 'absolute_prediabetes', 'absolute_no_diabetes', 'wrongly_diagnosed'], axis=1)
absolute_prediabetes_df = df[df['absolute_prediabetes']].drop(['absolute_diagnosed_diabetes', 'absolute_prediabetes', 'absolute_no_diabetes', 'wrongly_diagnosed'], axis=1)
absolute_no_diabetes_df = df[df['absolute_no_diabetes']].drop(['absolute_diagnosed_diabetes', 'absolute_prediabetes', 'absolute_no_diabetes', 'wrongly_diagnosed'], axis=1)
wrongly_diagnosed_df = df[df['wrongly_diagnosed']].drop(['absolute_diagnosed_diabetes', 'absolute_prediabetes', 'absolute_no_diabetes', 'wrongly_diagnosed'], axis=1)

# Save to CSV files
absolute_diagnosed_diabetes_df.to_csv(os.path.join(output_dir, 'absolute_diagnosed_diabetes.csv'), index=False)
absolute_prediabetes_df.to_csv(os.path.join(output_dir, 'absolute_prediabetes.csv'), index=False)
absolute_no_diabetes_df.to_csv(os.path.join(output_dir, 'absolute_no_diabetes.csv'), index=False)
wrongly_diagnosed_df.to_csv(os.path.join(output_dir, 'wrongly_diagnosed.csv'), index=False)

# Print summary
print("Categorization complete!")
print(f"Absolute Diagnosed Diabetes: {len(absolute_diagnosed_diabetes_df)} records")
print(f"Absolute Pre-Diabetes: {len(absolute_prediabetes_df)} records")
print(f"Absolute No Diabetes: {len(absolute_no_diabetes_df)} records")
print(f"Wrongly Diagnosed: {len(wrongly_diagnosed_df)} records")
print(f"Total records: {len(df)}")
print(f"Output directory: {output_dir}")

Categorization complete!
Absolute Diagnosed Diabetes: 1128 records
Absolute Pre-Diabetes: 626 records
Absolute No Diabetes: 2173 records
Wrongly Diagnosed: 2867 records
Total records: 21187
Output directory: categorized_diabetes


In [1]:
# Merge absolute diagnosed diabetes and absolute prediabetes CSV files
import pandas as pd
import os

# Load the categorized CSV files
diabetes_df = pd.read_csv('categorized_diabetes/absolute_diagnosed_diabetes.csv')
prediabetes_df = pd.read_csv('categorized_diabetes/absolute_prediabetes.csv')

# Merge the DataFrames
merged_diabetes_df = pd.concat([diabetes_df, prediabetes_df], ignore_index=True)

# Save the merged file
output_file = 'categorized_diabetes/merged_absolute_diabetes.csv'
merged_diabetes_df.to_csv(output_file, index=False)

print(f"Merged {len(diabetes_df)} diagnosed diabetes records with {len(prediabetes_df)} prediabetes records")
print(f"Total merged records: {len(merged_diabetes_df)}")
print(f"Saved merged file to: {output_file}")

Merged 1128 diagnosed diabetes records with 626 prediabetes records
Total merged records: 1754
Saved merged file to: categorized_diabetes/merged_absolute_diabetes.csv
